# 01 — Lakes and labels

**Goal (SPECS §4, PRD FR-1/FR-2):** build the two label tables everything else depends on.

| Output | What it is |
| --- | --- |
| `lakes.csv` | one row per glacial lake in the Hunza, Shimshal and Shigar basins (SPECS §3 schema) |
| `events.csv` | one row per GLOF event matched to a lake (SPECS §3 schema) |
| `events_unmatched.csv` | every database record that did **not** become a label, with the reason |
| `lakes_excluded.csv` | every water body that was **not** kept as a lake, with the reason |
| `lakes_polygons.geojson`, `basins.geojson` | geometries for notebook 02 and the map |
| `fig_01_basins_lakes.png`, `run_info.json` | map figure and a record of versions, parameters and counts |

**Steps:** 1 basins from HydroBASINS → 2 glacier outlines (RGI v7) → 3 lakes from JRC Global Surface Water →
4 shape filter, exclusions and Tier 2 lakes → 5 lake IDs → 6 events from the GLOF database → 7 Gate 1 checks → 8 save → 9 map.

**How to run:** Runtime → Run all. Approve the Earth Engine and Drive pop-ups.
Then send back the printed outputs of the cells marked **📋 REPORT** and the figure.

## 0. Setup
Clones the project repository so the notebook uses the **same tested code** as the repo (`src/karaglof/`),
instead of copying logic into cells. `odfpy` reads the GLOF database spreadsheet (`.ods`).

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/inamjadoon/KaraGlof.git"
REPO_DIR = "/content/KaraGlof"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "odfpy==1.4.1"], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
import json
import math
import time
import urllib.request
from datetime import UTC, datetime
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import drive

from karaglof import basins as kb
from karaglof import labels as kl

ee.Authenticate()
ee.Initialize(project="glof-497313")

drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/KaraGlof/01_lakes")
OUT.mkdir(parents=True, exist_ok=True)
EXPORT_FOLDER = "KaraGlof_exports"   # used only if Earth Engine needs a batch export
print("outputs ->", OUT)

## Parameters
Every choice that shapes the tables is here, in one place, so it can be reported and defended.

**Anchor points.** Basins are found from points that are surely inside or surely outside each basin
(see step 1). All anchors except Sost come straight from GLOF-database coordinates.

In [ ]:
# (lon, lat)
ANCHORS = {
    "shisper":   (74.562535, 36.360999),  # Shisper (Shisphare) glacier, Hunza  — GLOF database
    "passu":     (74.879298, 36.457205),  # Passu glacier lake, Hunza           — GLOF database
    "khurdopin": (75.478606, 36.342335),  # Khurdopin glacier, Shimshal         — GLOF database
    "biafo":     (75.910000, 35.690000),  # Biafo glacier, Braldu -> Shigar     — GLOF database
    "chihnji":   (74.004089, 36.783588),  # Karambar -> Ishkoman -> Gilgit river (NOT Hunza)
    "gargoh":    (74.665676, 35.997077),  # Bagrot valley -> Gilgit river      (NOT Hunza)
    "lokpar":    (76.270137, 35.495623),  # Hushe -> Shyok                     (NOT Shigar)
    "sost":      (74.820000, 36.690000),  # Sost, Hunza main stem above Shimshal (NOT Shimshal)
}
# basin: (start anchor, anchors that must stay outside)
BASIN_RULES = {
    "shimshal":  ("khurdopin", ["sost", "passu"]),
    "hunza_all": ("shisper",   ["chihnji", "gargoh"]),   # Hunza including Shimshal
    "shigar":    ("biafo",     ["lokpar"]),
}
# Rough published catchment sizes, used only as a sanity check (warn, not fail).
EXPECTED_KM2 = {"hunza_all": 13_900, "shigar": 6_900}
BBOX = [73.5, 35.0, 77.2, 37.3]          # lon_min, lat_min, lon_max, lat_max: must contain all three basins

MIN_LAKE_KM2 = 0.01                      # PRD §4: lakes smaller than 0.01 km² are dropped
MIN_PIXELS = 12                          # 12 × 30 m × 30 m = 0.0108 km² ≥ MIN_LAKE_KM2
GLACIER_MAX_DIST_M = 10_000              # a "glacial lake" lies within 10 km of a glacier (common inventory rule)
MAX_LAKE_KM2 = 20                        # larger water bodies here are rivers or Attabad, not glacial lakes
MAX_COMPACTNESS = 25                     # perimeter² / (4π·area): 1 = circle; river reaches are far higher
EXCLUDE_POINTS = {"Attabad": (74.8667, 36.3333)}   # PRD §4: landslide-dammed lake, never a GLOF lake
EXCLUDE_RADIUS_M = 1_500

TIER2 = {"Shisper": "shisper", "Khurdopin": "khurdopin", "Passu": "passu"}   # name -> anchor
TIER2_SEARCH_M = 3_000
TIER2_OVERRIDE = {}                      # e.g. {"Passu": 123}: candidate cand_id, ONLY after checking the map
EVENT_MATCH_RADIUS_M = 3_000             # database points often sit on the glacier/dam, not in the lake
UTM = "EPSG:32643"                       # UTM zone 43N: metric distances and areas for this region

## Helper: bring Earth Engine results into Python
Small results come back directly. If Earth Engine times out, the same collection is exported to Drive and read
from there, so the notebook still runs top to bottom.

In [ ]:
def fetch(fc, name):
    try:
        feats = fc.getInfo()["features"]
        return gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326")
    except ee.EEException as e:
        print(f"{name}: direct download failed ({str(e)[:120]}) -> exporting to Drive")
    task = ee.batch.Export.table.toDrive(collection=fc, description=f"karaglof_{name}",
                                         folder=EXPORT_FOLDER, fileNamePrefix=name,
                                         fileFormat="GeoJSON")
    task.start()
    while task.active():
        time.sleep(20)
        print("   ", name, task.status()["state"])
    if task.status()["state"] != "COMPLETED":
        raise RuntimeError(task.status())
    path = Path("/content/drive/MyDrive") / EXPORT_FOLDER / f"{name}.geojson"
    for _ in range(60):            # Drive can take a moment to show the new file
        if path.exists():
            break
        time.sleep(10)
    return gpd.read_file(path)

## 1. Basins from HydroBASINS (level 8)
HydroBASINS splits the land into small sub-basins; each one knows the sub-basin it drains into (`NEXT_DOWN`).
`karaglof.basins.delineate` walks downstream from a start anchor (e.g. Khurdopin glacier) and stops **just before**
the upstream area would include an anchor that is surely in the neighbouring valley (e.g. Sost on the Hunza main
stem). The result is the full basin above that point, with no hand drawing.
`hunza` = Hunza including Shimshal, **minus** Shimshal, so every lake belongs to exactly one basin.

Two automatic checks: the outlet's own `UP_AREA` must equal the sum of the sub-basin areas (otherwise `BBOX` cut the
basin off), and the areas should be close to published catchment sizes.

In [ ]:
hb = ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_8").filterBounds(ee.Geometry.Rectangle(BBOX))
attrs = hb.select(["HYBAS_ID", "NEXT_DOWN", "SUB_AREA", "UP_AREA"], None, False).getInfo()["features"]
hyb = pd.DataFrame([f["properties"] for f in attrs]).astype({"HYBAS_ID": "int64", "NEXT_DOWN": "int64"})
next_down = dict(zip(hyb.HYBAS_ID, hyb.NEXT_DOWN, strict=True))
sub_area = dict(zip(hyb.HYBAS_ID, hyb.SUB_AREA, strict=True))
up_area = dict(zip(hyb.HYBAS_ID, hyb.UP_AREA, strict=True))
print(len(hyb), "HydroBASINS level-8 sub-basins in the box")

def basin_of(anchor):
    hid = hb.filterBounds(ee.Geometry.Point(ANCHORS[anchor])).first().get("HYBAS_ID").getInfo()
    if hid is None:
        raise ValueError(f"anchor {anchor} is not inside any sub-basin")
    return int(hid)

anchor_basin = {a: basin_of(a) for a in ANCHORS}
basin_ids = {}
for name, (start, excl) in BASIN_RULES.items():
    outlet, ids = kb.delineate(next_down, anchor_basin[start], {anchor_basin[a] for a in excl})
    total = sum(sub_area[i] for i in ids)
    print(f"{name:10s} outlet {outlet}  {len(ids):3d} sub-basins  {total:8.0f} km²  "
          f"(outlet UP_AREA {up_area[outlet]:.0f} km²)")
    assert abs(total - up_area[outlet]) < 0.02 * up_area[outlet], f"{name}: BBOX cuts the basin — enlarge BBOX"
    if name in EXPECTED_KM2 and abs(total / EXPECTED_KM2[name] - 1) > 0.25:
        print(f"   ⚠ {name} differs from the expected ~{EXPECTED_KM2[name]} km² — check the map in step 9")
    basin_ids[name] = ids

basin_ids["hunza"] = basin_ids.pop("hunza_all") - basin_ids["shimshal"]
assert anchor_basin["khurdopin"] in basin_ids["shimshal"] and anchor_basin["shisper"] in basin_ids["hunza"]
assert anchor_basin["passu"] in basin_ids["hunza"]

def basin_geom(ids):
    return hb.filter(ee.Filter.inList("HYBAS_ID", sorted(ids))).union(maxError=30).first().geometry()

basins_fc = ee.FeatureCollection([
    ee.Feature(basin_geom(ids), {"basin": name, "area_km2": round(sum(sub_area[i] for i in ids))})
    for name, ids in basin_ids.items()
])
study = basins_fc.geometry()
basins_gdf = fetch(basins_fc.map(lambda f: f.simplify(maxError=50)), "basins")
print(basins_gdf[["basin", "area_km2"]].to_string(index=False))   # 📋 REPORT

## 2. Glacier outlines (parent glaciers)
SPECS asks for RGI v7. It is read from the GEE community catalog; if that asset cannot be read, the notebook falls
back to the official GLIMS collection (glacier boundaries only) and **says so** in `run_info.json`.

In [ ]:
def load_glaciers(region):
    try:
        fc = ee.FeatureCollection("projects/sat-io/open-datasets/RGI/RGI_VECTOR_MERGED_V7").filterBounds(region)
        if fc.size().getInfo() > 0:
            return fc, "RGI v7.0 (projects/sat-io/open-datasets/RGI/RGI_VECTOR_MERGED_V7)"
    except ee.EEException as e:
        print("RGI v7 not readable:", str(e)[:150])
    fc = ee.FeatureCollection("GLIMS/current").filterBounds(region).filter(ee.Filter.eq("line_type", "glac_bound"))
    return fc, "GLIMS/current (fallback, glac_bound outlines)"

glaciers, GLACIER_SOURCE = load_glaciers(study)
props = glaciers.first().propertyNames().getInfo()
GLACIER_ID = next(p for p in ["rgi_id", "RGIId", "glac_id"] if p in props)
print(GLACIER_SOURCE, "| id field:", GLACIER_ID, "| glaciers in study area:", glaciers.size().getInfo())

## 3. Candidate lakes from JRC Global Surface Water
**Why this source:** `max_extent` marks every 30 m pixel that was water at least once in 1984–2021 (Pekel et al.
2016, Landsat). Static lake inventories map one moment in time, but ice-dammed lakes such as Shisper and Khurdopin
**only exist during surges**, so a single-date inventory can miss exactly the lakes that matter here. `max_extent`
keeps them.

Steps: keep water patches of at least `MIN_PIXELS` connected pixels → turn them into polygons → measure area,
perimeter and centroid → keep only patches within `GLACIER_MAX_DIST_M` of a glacier and record that nearest glacier
as the **parent glacier**.

In [ ]:
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("max_extent")
water = gsw.eq(1).selfMask().clip(study)
water = water.updateMask(water.connectedPixelCount(MIN_PIXELS, True).gte(MIN_PIXELS))

polys = water.reduceToVectors(geometry=study, scale=30, geometryType="polygon", eightConnected=True,
                              labelProperty="water", maxPixels=1e10, tileScale=4)

def measure(f):
    g = f.geometry()
    c = g.centroid(maxError=10).coordinates()
    return f.set({"area_km2": g.area(maxError=10).divide(1e6),
                  "perimeter_km": g.perimeter(maxError=10).divide(1e3),
                  "lon": c.get(0), "lat": c.get(1)})

polys = polys.map(measure).filter(ee.Filter.gte("area_km2", MIN_LAKE_KM2))

near = ee.Filter.withinDistance(distance=GLACIER_MAX_DIST_M, leftField=".geo", rightField=".geo", maxError=30)
joined = ee.Join.saveBest(matchKey="glacier", measureKey="glacier_dist_m").apply(polys, glaciers, near)
joined = joined.map(lambda f: f.set("glacier_id", ee.Feature(f.get("glacier")).get(GLACIER_ID)))
joined = joined.select(["area_km2", "perimeter_km", "lon", "lat", "glacier_id", "glacier_dist_m"])

n_water = polys.size().getInfo()
cand = fetch(joined, "lake_candidates")
cand = cand.sort_values(["lat", "lon"], ascending=[False, True]).reset_index(drop=True)
cand["cand_id"] = cand.index
print(f"{n_water} water patches ≥ {MIN_LAKE_KM2} km²; {len(cand)} within {GLACIER_MAX_DIST_M/1000:.0f} km of a glacier")

## 4. Shape filter, excluded sites and the Tier 2 lakes
* **Basin:** each candidate takes the basin that contains its centroid.
* **Shape:** rivers also show up as water. A river reach is long and thin, so its *compactness*
  (perimeter² / 4π·area; 1 for a circle) is very high. Candidates above `MAX_COMPACTNESS` or larger than
  `MAX_LAKE_KM2` are excluded **with the reason written down**, so the choice can be reviewed.
* **Excluded sites (PRD §4):** Attabad is a landslide-dammed lake and is removed by location. (Gayari lies outside
  these basins.)
* **Tier 2:** Shisper, Khurdopin and Passu are the candidates closest to their GLOF-database points (within
  `TIER2_SEARCH_M`). They are kept even if the shape filter would drop them, and flagged so you can check them.

In [ ]:
cand_utm = cand.to_crs(UTM)
basins_utm = basins_gdf.to_crs(UTM)
cand["basin"] = gpd.sjoin(gpd.GeoDataFrame(geometry=cand_utm.geometry.centroid, crs=UTM),
                          basins_utm[["basin", "geometry"]], how="left", predicate="within")["basin"].values
cand["compactness"] = cand.perimeter_km ** 2 / (4 * math.pi * cand.area_km2)

reason = pd.Series("", index=cand.index)
reason[cand.basin.isna()] = "centroid outside the three basins"
reason[(reason == "") & (cand.area_km2 > MAX_LAKE_KM2)] = f"area > {MAX_LAKE_KM2} km² (river or non-glacial lake)"
reason[(reason == "") & (cand.compactness > MAX_COMPACTNESS)] = f"compactness > {MAX_COMPACTNESS} (river-like shape)"
for site, lonlat in EXCLUDE_POINTS.items():
    pt = gpd.GeoSeries(gpd.points_from_xy([lonlat[0]], [lonlat[1]]), crs="EPSG:4326").to_crs(UTM).iloc[0]
    hit = cand_utm.geometry.distance(pt) <= EXCLUDE_RADIUS_M
    reason[hit] = f"excluded site: {site} (PRD §4)"
    print(f"{site}: {int(hit.sum())} candidate(s) removed")
cand["exclude_reason"] = reason

tier2_pick = {}
for name, anchor in TIER2.items():
    pt = gpd.GeoSeries(gpd.points_from_xy(*[[v] for v in ANCHORS[anchor]]), crs="EPSG:4326").to_crs(UTM).iloc[0]
    dist = cand_utm.geometry.distance(pt)
    if name in TIER2_OVERRIDE:
        pick = TIER2_OVERRIDE[name]
    else:
        near_ok = dist[(dist <= TIER2_SEARCH_M) & (cand.exclude_reason == "")]
        near_any = dist[dist <= TIER2_SEARCH_M]
        pick = (near_ok if len(near_ok) else near_any).idxmin() if len(near_any) else None
    if pick is None:
        raise ValueError(f"No candidate within {TIER2_SEARCH_M} m of {name} — check the map")
    print(f"{name:10s} -> cand_id {pick}: {cand.area_km2[pick]:.3f} km², {dist[pick]:.0f} m from the database point, "
          f"compactness {cand.compactness[pick]:.1f}" + (f"  ⚠ kept although: {cand.exclude_reason[pick]}"
                                                          if cand.exclude_reason[pick] else ""))   # 📋 REPORT
    print("            other candidates within 3 km:",
          dist[(dist <= TIER2_SEARCH_M) & (dist.index != pick)].round(0).to_dict())
    tier2_pick[name] = pick
    cand.loc[pick, "exclude_reason"] = ""

print(cand.exclude_reason.replace("", "KEPT").value_counts().to_string())   # 📋 REPORT

## 5. Lake table and stable IDs
IDs follow `KG-<basin>-<number>` (`HUN`, `SHM`, `SHG`), numbered north → south inside each basin
(`karaglof.labels.assign_lake_ids`), so the same inventory always gets the same IDs.
`area_ref_km2` is the **maximum extent 1984–2021**, the area reference for this inventory.

In [ ]:
kept = cand[cand.exclude_reason == ""].copy()
kept["name"] = ""
kept["tier2"] = False
for name, pick in tier2_pick.items():
    kept.loc[pick, ["name", "tier2"]] = [name, True]
kept["dam_type"] = np.where(kept.tier2, "ice", "unknown")   # refined from the event records in step 6
lakes_gdf = gpd.GeoDataFrame(kl.assign_lake_ids(kept), geometry="geometry", crs="EPSG:4326")
lakes_gdf["area_ref_km2"] = lakes_gdf.area_km2.round(4)
print(lakes_gdf.groupby("basin").size().to_string())   # 📋 REPORT
print(lakes_gdf[lakes_gdf.tier2][["lake_id", "name", "area_ref_km2", "glacier_id"]].to_string(index=False))

## 6. Events from the GLOF database (Lützow & Veh, v4.2)
The database (Zenodo, CC BY 4.0) is downloaded once and cached in Drive. `karaglof.labels` then:
* drops the sheet's description rows and cleans the text;
* parses partial dates (`2018`, `2018-04`, `2018-04-10`) and records the **date certainty** (`year` / `month` / `day`);
* keeps a record only if it is inside the three basins, dated 2000–2025, and a real lake
  (englacial **water pockets** have no lake to rank) and not an excluded site;
* matches each record to the nearest lake polygon within `EVENT_MATCH_RADIUS_M`.

Every record that is not used goes to `events_unmatched.csv` **with its reason**, which is the Gate 1 rule.

In [ ]:
db_path = OUT / "raw" / kl.GLOF_DB_FILE
db_path.parent.mkdir(exist_ok=True)
if not db_path.exists():
    urllib.request.urlretrieve(kl.GLOF_DB_URL, db_path)
db = kl.load_glof_database(db_path)

pts = gpd.GeoDataFrame(db, geometry=gpd.points_from_xy(db.lon, db.lat), crs="EPSG:4326").to_crs(UTM)
inside = pd.Series(pts.geometry.within(basins_utm.union_all()).values, index=db.index)
prepared = kl.prepare_events(db, inside)

lakes_utm = lakes_gdf.to_crs(UTM)
dist = pd.DataFrame({lid: pts.geometry.distance(g) for lid, g in zip(lakes_utm.lake_id, lakes_utm.geometry, strict=True)},
                    index=db.index)
matched = kl.match_events(prepared, dist, EVENT_MATCH_RADIUS_M)
events = kl.build_events_table(matched)
unmatched = matched[matched.lake_id == ""]

# dam type from the records of each lake (most common type); Tier 2 lakes keep a recorded type too
dam = matched[matched.lake_id != ""].groupby("lake_id").dam_type.agg(lambda s: s.mode().iloc[0])
lakes_gdf["dam_type"] = lakes_gdf.lake_id.map(dam).fillna(lakes_gdf.dam_type)

print(f"{len(db)} records in {kl.GLOF_DB_SHEET}; {len(events)} events matched; {len(unmatched)} not used")   # 📋 REPORT
print(unmatched.exclude_reason.str.replace(r"\(nearest \d+ m\)", "", regex=True).value_counts().to_string())

## 7. Gate 1 checks 📋 REPORT
`karaglof.labels.check_labels` must return **no problems** before notebook 02. Below the checks, the number of
positive lake-years in each model split (train 2000–2017, validation 2018–2019, test 2020–2025). That count
decides how much the Tier 1 model can learn.

In [ ]:
problems = kl.check_labels(lakes_gdf, events, unmatched, list(TIER2))
print("Gate 1:", "PASS" if not problems else "FAIL")
for p in problems:
    print("  -", p)

ev = events.assign(year=events.date.str[:4].astype(int))
ev = ev.merge(lakes_gdf[["lake_id", "name", "basin"]], on="lake_id")
print("\nEvents per lake:")
print(ev.groupby(["lake_id", "name", "basin"]).agg(events=("event_id", "size"),
      years=("year", lambda y: sorted(set(y)))).to_string())
pos = ev.drop_duplicates(["lake_id", "year"]).copy()
pos["split"] = pd.cut(pos.year, [1999, 2017, 2019, 2025], labels=["train", "validation", "test"])
print("\nPositive lake-years per split:")
print(pos.groupby("split", observed=False).agg(lake_years=("year", "size"),
      lakes=("lake_id", "nunique")).to_string())
print(f"\nLakes: {len(lakes_gdf)}  -> lake-years 2000-2025: {len(lakes_gdf) * 26}; "
      f"positive share ≈ {len(pos) / (len(lakes_gdf) * 26):.4f}")
assert not problems, "Gate 1 failed — report the problems above"

## 8. Save outputs to Drive
`lakes.csv` and `events.csv` follow the SPECS §3 columns exactly. `run_info.json` records data sources, versions,
parameters and counts, so the run can be reproduced and cited.

In [ ]:
LAKE_COLS = ["lake_id", "name", "basin", "lat", "lon", "area_ref_km2", "dam_type", "glacier_id", "tier2"]
lakes_out = lakes_gdf.copy()
lakes_out[["lat", "lon"]] = lakes_out[["lat", "lon"]].round(5)
lakes_out[LAKE_COLS].to_csv(OUT / "lakes.csv", index=False)
events.to_csv(OUT / "events.csv", index=False)
unmatched[["db_id", "glacier", "lake_name", "lake_type", "lat", "lon", "date_text", "date_certainty",
           "date_min", "date_max", "match_dist_m", "exclude_reason"]].to_csv(OUT / "events_unmatched.csv", index=False)
cand[cand.exclude_reason != ""].drop(columns="geometry").to_csv(OUT / "lakes_excluded.csv", index=False)
lakes_gdf[LAKE_COLS + ["geometry"]].to_file(OUT / "lakes_polygons.geojson", driver="GeoJSON")
basins_gdf.to_file(OUT / "basins.geojson", driver="GeoJSON")

run_info = {
    "notebook": "01_lakes_and_labels", "run_utc": datetime.now(UTC).isoformat(timespec="seconds"),
    "sources": {"basins": "WWF/HydroSHEDS/v1/Basins/hybas_8", "water": "JRC/GSW1_4/GlobalSurfaceWater max_extent",
                "glaciers": GLACIER_SOURCE, "events": kl.GLOF_DB_SOURCE},
    "parameters": {k: v for k, v in globals().items() if k.isupper() and k not in ("OUT", "REPO_DIR")
                   and isinstance(v, int | float | str | list | dict)},
    "counts": {"water_patches": n_water, "candidates": len(cand), "lakes": len(lakes_gdf),
               "lakes_per_basin": lakes_gdf.basin.value_counts().to_dict(), "events": len(events),
               "unmatched_records": len(unmatched), "positive_lake_years": len(pos)},
    "tier2": {n: lakes_gdf.loc[lakes_gdf.name == n, "lake_id"].item() for n in TIER2},
    "versions": {"earthengine-api": ee.__version__, "geemap": geemap.__version__,
                 "pandas": pd.__version__, "geopandas": gpd.__version__},
}
(OUT / "run_info.json").write_text(json.dumps(run_info, indent=2, default=str))
print(json.dumps(run_info["counts"], indent=2), json.dumps(run_info["tier2"], indent=2))   # 📋 REPORT
print(sorted(p.name for p in OUT.iterdir()))

## 9. Map check 📋 REPORT (figure)
**Look at three things:** (1) do the three basin outlines follow the real valleys? (2) are the red Tier 2 lakes on
Shisper, Khurdopin and Passu? (3) are there obvious river reaches still kept as lakes? Click a lake to see its
properties. The static figure is saved to Drive for the write-up.

In [ ]:
m = geemap.Map(center=(36.0, 75.3), zoom=8)
m.add_basemap("Esri.WorldImagery")
m.addLayer(gsw.selfMask().clip(study), {"palette": ["00bfff"]}, "GSW max extent", False)
m.add_gdf(basins_gdf, layer_name="basins", style={"color": "yellow", "fillOpacity": 0})
m.add_gdf(lakes_gdf[~lakes_gdf.tier2][LAKE_COLS + ["geometry"]], layer_name="lakes",
          style={"color": "cyan", "fillOpacity": 0.4})
m.add_gdf(lakes_gdf[lakes_gdf.tier2][LAKE_COLS + ["geometry"]], layer_name="Tier 2 lakes",
          style={"color": "red", "fillOpacity": 0.6})
used = matched[matched.lake_id != ""]
m.add_gdf(gpd.GeoDataFrame(used[["glacier", "lake_id"]], crs="EPSG:4326",
                           geometry=gpd.points_from_xy(used.lon, used.lat)), layer_name="GLOF events")
m

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors = {"hunza": "#4c78a8", "shimshal": "#f58518", "shigar": "#54a24b"}
for _, b in basins_gdf.iterrows():
    gpd.GeoSeries([b.geometry]).boundary.plot(ax=ax, color=colors[b.basin], lw=1.2)
    ax.annotate(b.basin.title(), b.geometry.representative_point().coords[0], color=colors[b.basin], fontsize=11)
ax.scatter(lakes_gdf.lon, lakes_gdf.lat, s=4 + 60 * np.sqrt(lakes_gdf.area_ref_km2), c="#1f77b4",
           alpha=0.6, label=f"glacial lakes (n={len(lakes_gdf)})")
t2 = lakes_gdf[lakes_gdf.tier2]
ax.scatter(t2.lon, t2.lat, s=90, facecolors="none", edgecolors="red", lw=2, label="Tier 2 lakes")
for _, r in t2.iterrows():
    ax.annotate(f"{r['name']}\n{r.lake_id}", (r.lon, r.lat), xytext=(6, 6), textcoords="offset points", fontsize=8)
ax.scatter(used.lon, used.lat, marker="x", c="k", s=30, label=f"GLOF records 2000-2025 (n={len(used)})")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.set_aspect(1 / math.cos(math.radians(36)))
ax.set_title("KaraGLOF study basins, glacial lakes and GLOF records")
ax.legend(loc="lower left", fontsize=8)
fig.savefig(OUT / "fig_01_basins_lakes.png", dpi=200, bbox_inches="tight")
plt.show()

## What to send back
1. The output of every cell marked **📋 REPORT** (copy the text).
2. `fig_01_basins_lakes.png` (or a screenshot of the interactive map).
3. Anything that looks wrong on the map: a basin outline off the valley, a Tier 2 lake in the wrong place,
   or rivers kept as lakes.

Do **not** edit thresholds yourself to "make it look better". Report it and we'll change it in the repo with a
recorded reason, so every choice stays defensible.